# Membaca satu request di `spans.json`

Notebook ini **berdiri sendiri**: datanya sudah ditanam di dalam, jadi bisa langsung **Run all** di Colab tanpa mengunggah dataset.

**Request yang dipakai: login** (`POST /api/v1/users/login`) pada aplikasi TrainTicket. Hanya **3 span** dan **2 service**:

1. `ts-auth-service` menerima login. Ini span akar, `getToken`.
2. Sebelum memberi token, ia memanggil `ts-verification-code-service` untuk memeriksa kode captcha (`verifyCode`).

Sumber: run `TT.2022-04-17T212101D2022-04-17T230842`, `traceID = 10ffc29932ddc2c7`.

## 1. Data mentah

Sel di bawah memuat **satu elemen** dari list di `spans.json`, apa adanya. Satu elemen = satu *trace* = satu request.

In [1]:
import json, datetime

# Satu elemen dari list di spans.json. Disimpan dalam satu baris panjang; tampilan rapinya ada di sel berikutnya.
ENTRI = json.loads(r'''{"traceID":"10ffc29932ddc2c7","spans":[{"traceID":"10ffc29932ddc2c7","spanID":"3318f59a84ac5d5d","flags":1,"operationName":"GET","references":[{"refType":"CHILD_OF","traceID":"10ffc29932ddc2c7","spanID":"10ffc29932ddc2c7"}],"startTime":1650201902285000,"duration":2586,"tags":[{"key":"http.status_code","type":"int64","value":200},{"key":"component","type":"string","value":"java-spring-rest-template"},{"key":"span.kind","type":"string","value":"client"},{"key":"http.url","type":"string","value":"http://ts-verification-code-service:15678/api/v1/verifycode/verify/AAAA"},{"key":"peer.port","type":"int64","value":15678},{"key":"http.method","type":"string","value":"GET"},{"key":"internal.span.format","type":"string","value":"proto"}],"logs":[],"processID":"p1","warnings":null},{"traceID":"10ffc29932ddc2c7","spanID":"10ffc29932ddc2c7","flags":1,"operationName":"getToken","references":[],"startTime":1650201902284000,"duration":124900,"tags":[{"key":"http.status_code","type":"int64","value":200},{"key":"component","type":"string","value":"java-web-servlet"},{"key":"span.kind","type":"string","value":"server"},{"key":"sampler.type","type":"string","value":"const"},{"key":"sampler.param","type":"bool","value":true},{"key":"http.url","type":"string","value":"http://ts-auth-service:12340/api/v1/users/login"},{"key":"http.method","type":"string","value":"POST"},{"key":"internal.span.format","type":"string","value":"proto"}],"logs":[{"timestamp":1650201902285000,"fields":[{"key":"event","type":"string","value":"preHandle"},{"key":"handler","type":"string","value":"public org.springframework.http.ResponseEntity<edu.fudan.common.util.Response> auth.controller.UserController.getToken(auth.dto.BasicAuthDto,org.springframework.http.HttpHeaders)"},{"key":"handler.class_simple_name","type":"string","value":"UserController"},{"key":"handler.method_name","type":"string","value":"getToken"}]},{"timestamp":1650201902409000,"fields":[{"key":"event","type":"string","value":"afterCompletion"},{"key":"handler","type":"string","value":"public org.springframework.http.ResponseEntity<edu.fudan.common.util.Response> auth.controller.UserController.getToken(auth.dto.BasicAuthDto,org.springframework.http.HttpHeaders)"}]}],"processID":"p1","warnings":null},{"traceID":"10ffc29932ddc2c7","spanID":"56643928cd3d0a72","flags":1,"operationName":"verifyCode","references":[{"refType":"CHILD_OF","traceID":"10ffc29932ddc2c7","spanID":"3318f59a84ac5d5d"}],"startTime":1650201902285645,"duration":1295,"tags":[{"key":"http.status_code","type":"int64","value":200},{"key":"http.url","type":"string","value":"http://ts-auth-service:12340/api/v1/verifycode/verify/AAAA"},{"key":"component","type":"string","value":"java-web-servlet"},{"key":"span.kind","type":"string","value":"server"},{"key":"http.method","type":"string","value":"GET"},{"key":"internal.span.format","type":"string","value":"proto"}],"logs":[{"timestamp":1650201902286645,"fields":[{"key":"event","type":"string","value":"preHandle"},{"key":"handler","type":"string","value":"public boolean verifycode.controller.VerifyCodeController.verifyCode(java.lang.String,javax.servlet.http.HttpServletRequest,javax.servlet.http.HttpServletResponse,org.springframework.http.HttpHeaders)"},{"key":"handler.class_simple_name","type":"string","value":"VerifyCodeController"},{"key":"handler.method_name","type":"string","value":"verifyCode"}]},{"timestamp":1650201902286645,"fields":[{"key":"event","type":"string","value":"afterCompletion"},{"key":"handler","type":"string","value":"public boolean verifycode.controller.VerifyCodeController.verifyCode(java.lang.String,javax.servlet.http.HttpServletRequest,javax.servlet.http.HttpServletResponse,org.springframework.http.HttpHeaders)"}]}],"processID":"p2","warnings":null}],"processes":{"p1":{"serviceName":"ts-auth-service","tags":[{"key":"hostname","type":"string","value":"a1d4b310b9e9"},{"key":"ip","type":"string","value":"172.26.0.7"},{"key":"jaeger.version","type":"string","value":"Java-0.30.6"}]},"p2":{"serviceName":"ts-verification-code-service","tags":[{"key":"hostname","type":"string","value":"4236cb019bd7"},{"key":"ip","type":"string","value":"172.26.0.8"},{"key":"jaeger.version","type":"string","value":"Java-0.30.6"}]}},"warnings":null}''')

In [2]:
def tampil(x, ind=0):
    """Cetak JSON secara ringkas: dict/objek kecil (mis. satu tag) cukup satu baris."""
    pad = "  " * ind
    skalar = lambda v: not isinstance(v, (dict, list))
    if isinstance(x, dict):
        if all(skalar(v) for v in x.values()):
            print(pad + json.dumps(x, ensure_ascii=False)); return
        for k, v in x.items():
            if skalar(v) or v == []:
                print(f"{pad}{k}: {json.dumps(v, ensure_ascii=False)}")
            else:
                print(f"{pad}{k}:"); tampil(v, ind + 1)
    elif isinstance(x, list):
        for i, v in enumerate(x):
            if isinstance(v, dict) and all(skalar(w) for w in v.values()):
                print(f"{pad}- " + json.dumps(v, ensure_ascii=False))
            else:
                print(f"{pad}[{i}]"); tampil(v, ind + 1)

tampil(ENTRI)

traceID: "10ffc29932ddc2c7"
spans:
  [0]
    traceID: "10ffc29932ddc2c7"
    spanID: "3318f59a84ac5d5d"
    flags: 1
    operationName: "GET"
    references:
      - {"refType": "CHILD_OF", "traceID": "10ffc29932ddc2c7", "spanID": "10ffc29932ddc2c7"}
    startTime: 1650201902285000
    duration: 2586
    tags:
      - {"key": "http.status_code", "type": "int64", "value": 200}
      - {"key": "component", "type": "string", "value": "java-spring-rest-template"}
      - {"key": "span.kind", "type": "string", "value": "client"}
      - {"key": "http.url", "type": "string", "value": "http://ts-verification-code-service:15678/api/v1/verifycode/verify/AAAA"}
      - {"key": "peer.port", "type": "int64", "value": 15678}
      - {"key": "http.method", "type": "string", "value": "GET"}
      - {"key": "internal.span.format", "type": "string", "value": "proto"}
    logs: []
    processID: "p1"
    warnings: null
  [1]
    traceID: "10ffc29932ddc2c7"
    spanID: "10ffc29932ddc2c7"
    flags: 1
   

### Cara membaca data mentah di atas

| Bagian | Artinya |
|---|---|
| `traceID` | ID request. Semua span request ini memakai ID yang sama |
| `processes` | kamus `kode proses -> service` (`p1` = `ts-auth-service`, `p2` = `ts-verification-code-service`), plus hostname dan IP container |
| `spans` | daftar span, di sini ada 3 |
| `spanID` | ID span. Kunci unik span = pasangan `(traceID, spanID)` |
| `references` | **induk** span. Kosong berarti span akar |
| `operationName` | nama operasi. Span *client* berisi metode HTTP (`GET`), span *server* berisi nama metode controller (`getToken`, `verifyCode`) |
| `startTime`, `duration` | dalam **mikrodetik** (epoch untuk `startTime`) |
| `processID` | kunci ke `processes`, yaitu service pemilik span |
| `tags` | metadata: `span.kind` (client/server), `http.url`, `http.method`, `http.status_code`, dll. |
| `logs` | event di dalam span (`preHandle` dan `afterCompletion`, hanya ada di span server) |

## 2. Diratakan jadi tabel

Satu baris per span. Waktu dibuat **relatif terhadap awal request** (milidetik) supaya mudah dibaca. ID dipendekkan jadi 6 karakter pertama.

In [3]:
proses = {k: v["serviceName"] for k, v in ENTRI["processes"].items()}
spans = ENTRI["spans"]

def tag(s, k):
    return next((t["value"] for t in s["tags"] if t["key"] == k), None)

def induk_of(s):
    return next((r["spanID"] for r in s["references"] if r["refType"] == "CHILD_OF"), None)

akar = next(s for s in spans if induk_of(s) is None)
t0 = akar["startTime"]
print("awal request (UTC):", datetime.datetime.fromtimestamp(t0 / 1e6, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3])
print()
print(f"{'span':7}{'induk':7}{'service':30}{'jenis':8}{'operasi':11}{'mulai(ms)':>10}{'durasi(ms)':>11}{'status':>7}")
for s in sorted(spans, key=lambda s: s["startTime"]):
    print(f"{s['spanID'][:6]:7}{(induk_of(s) or '-')[:6]:7}{proses[s['processID']]:30}{tag(s,'span.kind'):8}{s['operationName']:11}"
          f"{(s['startTime']-t0)/1e3:>10.3f}{s['duration']/1e3:>11.3f}{tag(s,'http.status_code'):>7}")

awal request (UTC): 2022-04-17 13:25:02.284

span   induk  service                       jenis   operasi     mulai(ms) durasi(ms) status
10ffc2 -      ts-auth-service               server  getToken        0.000    124.900    200
3318f5 10ffc2 ts-auth-service               client  GET             1.000      2.586    200
566439 3318f5 ts-verification-code-service  server  verifyCode      1.645      1.295    200


## 3. Pohon pemanggilan

Anak menunjuk induknya lewat `references`. Dengan itu kita bisa menggambar pohon.

In [4]:
anak = {}
for s in spans:
    anak.setdefault(induk_of(s), []).append(s)

def cetak(sid, d=0):
    for s in sorted(anak.get(sid, []), key=lambda s: s["startTime"]):
        print("    " * d + f"└─ [{proses[s['processID']]}] {s['operationName']}  ({tag(s,'span.kind')}, {s['duration']/1e3:.3f} ms)")
        cetak(s["spanID"], d + 1)

cetak(None)

└─ [ts-auth-service] getToken  (server, 124.900 ms)
    └─ [ts-auth-service] GET  (client, 2.586 ms)
        └─ [ts-verification-code-service] verifyCode  (server, 1.295 ms)


## 4. Linimasa

Posisi bar = kapan span mulai, panjang bar = durasinya (span yang sangat pendek minimal 1 blok).

In [5]:
total = akar["duration"]
LEBAR = 60
for s in sorted(spans, key=lambda s: s["startTime"]):
    a = round((s["startTime"] - t0) / total * LEBAR)
    b = max(1, round(s["duration"] / total * LEBAR))
    print(f"{s['operationName']:11}{tag(s,'span.kind'):7}|" + " " * a + "█" * b + f"  {s['duration']/1e3:.1f} ms")
print(" " * 18 + "0 ms" + " " * (LEBAR - 10) + f"{total/1e3:.0f} ms")

getToken   server |████████████████████████████████████████████████████████████  124.9 ms
GET        client |█  2.6 ms
verifyCode server | █  1.3 ms
                  0 ms                                                  125 ms


## 5. Apa yang bisa dihitung dari satu trace

Ketiga angka ini adalah dasar fitur latensi per service.

In [6]:
client = next(s for s in spans if tag(s, "span.kind") == "client")
server_anak = next(s for s in spans if induk_of(s) == client["spanID"])

print(f"Latensi total request          : {akar['duration']/1e3:8.3f} ms   (durasi span akar)")
print(f"Waktu menunggu service lain    : {client['duration']/1e3:8.3f} ms   (durasi span client ke {proses[server_anak['processID']]})")
print(f"Kerja di ts-auth-service sendiri: {(akar['duration']-client['duration'])/1e3:8.3f} ms   (akar - client)")
print(f"Selisih client vs server anak  : {(client['duration']-server_anak['duration'])/1e3:8.3f} ms   (jaringan + overhead)")
print()
print("Sisi graf dependensi:", proses[client["processID"]], "->", proses[server_anak["processID"]])

Latensi total request          :  124.900 ms   (durasi span akar)
Waktu menunggu service lain    :    2.586 ms   (durasi span client ke ts-verification-code-service)
Kerja di ts-auth-service sendiri:  122.314 ms   (akar - client)
Selisih client vs server anak  :    1.291 ms   (jaringan + overhead)

Sisi graf dependensi: ts-auth-service -> ts-verification-code-service


## 6. Hubungan ke `logs.json`

Request yang sama meninggalkan jejak di log. Baris `Span reported: <traceID>:<spanID>:<indukSpanID>:<flags> - <operasi>` memuat ID yang sama dengan span. Itulah kunci untuk menggabungkan **log** dengan **trace**. Induk `0` berarti span akar.

Tiga baris di bawah diambil dari `logs.json` run yang sama.

In [7]:
import re
LOG = [
    ('ts-auth-service', '2022-04-17 13:25:02.288  INFO 1 --- [io-12340-exec-7] i.j.internal.reporters.LoggingReporter   : Span reported: 10ffc29932ddc2c7:3318f59a84ac5d5d:10ffc29932ddc2c7:1 - GET'),
    ('ts-auth-service', '2022-04-17 13:25:02.409  INFO 1 --- [io-12340-exec-7] i.j.internal.reporters.LoggingReporter   : Span reported: 10ffc29932ddc2c7:10ffc29932ddc2c7:0:1 - getToken'),
    ('ts-verification-code-service', '2022-04-17 13:25:02.288  INFO 1 --- [io-15678-exec-9] i.j.internal.reporters.LoggingReporter   : Span reported: 10ffc29932ddc2c7:56643928cd3d0a72:3318f59a84ac5d5d:1 - verifyCode'),
]

ids_span = {s["spanID"] for s in spans}
print(f"{'service (kunci di logs.json)':30}{'jam log':14}{'spanID':18}{'induk':18}{'operasi':12}{'ada di trace?'}")
for svc, baris in LOG:
    jam = baris.split()[1]
    tid, sid, pid, flag, op = re.search(r"Span reported: (\w+):(\w+):(\w+):(\d+) - (\w+)", baris).groups()
    print(f"{svc:30}{jam:14}{sid:18}{pid:18}{op:12}{sid in ids_span and tid == ENTRI['traceID']}")

service (kunci di logs.json)  jam log       spanID            induk             operasi     ada di trace?
ts-auth-service               13:25:02.288  3318f59a84ac5d5d  10ffc29932ddc2c7  GET         True
ts-auth-service               13:25:02.409  10ffc29932ddc2c7  0                 getToken    True
ts-verification-code-service  13:25:02.288  56643928cd3d0a72  3318f59a84ac5d5d  verifyCode  True


## Hal yang perlu diingat

1. **Satu elemen di list = satu request (trace)**, bukan satu service. Nama service hanya label di tiap span lewat `processID -> processes`.
2. **Selalu ambil nama service dari `processes`**, bukan dari host di `http.url`. Di span server, host URL bisa menunjuk ke pemanggil.
3. **Satu panggilan antar service = sepasang span** (client di pemanggil, server di penerima). Pasangan itu membentuk sisi graf dependensi.
4. **Satuan waktu mikrodetik.** Bagi 1e6 untuk detik dan 1e3 untuk milidetik.
5. **Di file asli, elemen ini muncul 2 kali dan identik** (duplikasi). Di run ini 503 dari 817 `traceID` muncul lebih dari sekali, dan semuanya salinan persis (bukan potongan). Cukup dedup berdasarkan `(traceID, spanID)`, dan jangan menghitung jumlah request dari panjang list.
6. **Bentuk request itu terbatas.** Di run pertama, trace yang lengkap hanya berbentuk 1, 3, 8, 9, atau 23 span. Beban kerja ini berulang dan hampir deterministik.
7. Untuk menyejajarkan dengan label `fault.json` dan metrics, jam spans/logs harus disesuaikan dengan **offset 8 jam** (temuan di `EDA.ipynb` bagian 5).